# Engagement Detection System – DAiSEE feature extraction and baselines (Colab)
Run the cells in order. Every result is saved to your Google Drive folder `daisee_project`, so if Colab disconnects you can reconnect, re-run cells 1–4 and the extraction cell, and it **resumes** where it stopped.

Runtime: *Runtime → Change runtime type → CPU is enough for extraction.* Keep this tab open while extraction runs.

Nothing here stores frames, landmarks or identities: only per-clip numeric features.

## 1. Install and download the face model

In [ ]:
!pip -q install mediapipe xgboost shap
!wget -q -O /content/face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
!ls -la /content/face_landmarker.task

## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/daisee_project'
os.makedirs(WORK, exist_ok=True)
print('outputs go to', WORK)

## 3. Project code (written to the Colab disk)

In [ ]:
%%writefile /content/ed_features.py
"""Per-clip feature extraction for DAiSEE. No pixels are written to disk."""
import numpy as np, cv2

R_EYE = [33, 160, 158, 133, 153, 144]     # p1..p6, right eye (image left)
L_EYE = [362, 385, 387, 263, 373, 380]    # p1..p6, left eye
R_IRIS, L_IRIS = 468, 473
BLEND = ["eyeBlinkLeft", "eyeBlinkRight", "eyeLookInLeft", "eyeLookOutLeft", "eyeLookInRight",
         "eyeLookOutRight", "eyeLookUpLeft", "eyeLookDownLeft", "browDownLeft", "browDownRight",
         "browInnerUp", "jawOpen", "mouthSmileLeft", "mouthSmileRight"]
SIGNALS = ["yaw", "pitch", "roll", "ear", "gaze_x", "gaze_y", "motion"] + BLEND

def eye_aspect_ratio(eye):
    v1 = np.linalg.norm(eye[1] - eye[5]); v2 = np.linalg.norm(eye[2] - eye[4])
    h = np.linalg.norm(eye[0] - eye[3])
    return (v1 + v2) / (2.0 * h + 1e-8)

def head_pose_deg(M):
    R = M[:3, :3] / np.linalg.norm(M[:3, :3], axis=0)
    sy = np.hypot(R[0, 0], R[1, 0])
    pitch = np.degrees(np.arctan2(R[2, 1], R[2, 2]))
    yaw = np.degrees(np.arctan2(-R[2, 0], sy))
    roll = np.degrees(np.arctan2(R[1, 0], R[0, 0]))
    return yaw, pitch, roll

def clahe_if_dark(bgr, thresh=90, clip=2.0):
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    if lab[..., 0].mean() >= thresh:
        return bgr, False
    lab[..., 0] = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8, 8)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR), True

def frame_signals(res, w, h):
    """res: FaceLandmarkerResult. Returns dict of signals or None if no face."""
    if not res.face_landmarks:
        return None
    lm = res.face_landmarks[0]
    pts = np.array([[p.x * w, p.y * h] for p in lm])
    out = {}
    out["yaw"], out["pitch"], out["roll"] = head_pose_deg(np.asarray(res.facial_transformation_matrixes[0]))
    out["ear"] = 0.5 * (eye_aspect_ratio(pts[R_EYE]) + eye_aspect_ratio(pts[L_EYE]))
    gx, gy = [], []
    for iris, (a, b, up, lo) in ((R_IRIS, (33, 133, 159, 145)), (L_IRIS, (362, 263, 386, 374))):
        c = 0.5 * (pts[a] + pts[b]); wid = np.linalg.norm(pts[a] - pts[b]) + 1e-8
        gx.append((pts[iris][0] - c[0]) / wid)
        gy.append((pts[iris][1] - 0.5 * (pts[up][1] + pts[lo][1])) / wid)
    out["gaze_x"], out["gaze_y"] = float(np.mean(gx)), float(np.mean(gy))
    bs = {c.category_name: c.score for c in res.face_blendshapes[0]} if res.face_blendshapes else {}
    for k in BLEND:
        out[k] = float(bs.get(k, np.nan))
    x0, y0 = np.clip(pts.min(0), 0, [w - 1, h - 1]); x1, y1 = np.clip(pts.max(0), 0, [w - 1, h - 1])
    out["_bbox"] = (int(x0), int(y0), int(x1), int(y1))
    return out

def face_gray(bgr, bbox, size=64):
    x0, y0, x1, y1 = bbox
    if x1 - x0 < 8 or y1 - y0 < 8:
        return None
    return cv2.resize(cv2.cvtColor(bgr[y0:y1, x0:x1], cv2.COLOR_BGR2GRAY), (size, size))

def median3(a):
    a = np.asarray(a, float)
    if len(a) < 3:
        return a
    out = a.copy()
    for i in range(1, len(a) - 1):
        win = a[i - 1:i + 2]; win = win[~np.isnan(win)]
        out[i] = np.median(win) if len(win) else np.nan
    return out

def extract_clip(path, landmarker, target_fps=5):
    """Return (T, len(SIGNALS)) array with NaN rows where no face; plus n_frames and n_clahe."""
    import mediapipe as mp
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    step = max(1, int(round(fps / target_fps)))
    rows, prev, i, n_clahe = [], None, 0, 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i % step == 0:
            frame, used = clahe_if_dark(frame); n_clahe += used
            h, w = frame.shape[:2]
            res = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
            sig = frame_signals(res, w, h)
            if sig is None:
                rows.append([np.nan] * len(SIGNALS)); prev = None
            else:
                g = face_gray(frame, sig["_bbox"])
                motion = np.nan
                if g is not None and prev is not None:
                    flow = cv2.calcOpticalFlowFarneback(prev, g, None, 0.5, 3, 15, 3, 5, 1.2, 0)
                    motion = float(np.linalg.norm(flow, axis=2).mean())
                prev = g
                sig["motion"] = motion
                rows.append([sig[k] for k in SIGNALS])
        i += 1
    cap.release()
    A = np.array(rows, float).reshape(-1, len(SIGNALS))
    det = ~np.isnan(A[:, 0]) if len(A) else np.zeros(0, bool)   # raw detections, before filtering
    for j in range(A.shape[1]):
        A[:, j] = median3(A[:, j])
    return A, det, n_clahe

def window_features(A, det, ear_closed=0.2):
    """One row of features for one 10 s clip (the whole clip is one window). det = raw detection mask."""
    T = len(A)
    feats = {"n_frames": T, "detect_rate": float(det.mean()) if T else 0.0}
    t = np.arange(T)
    for j, name in enumerate(SIGNALS):
        x = A[:, j] if T else np.array([])
        m = ~np.isnan(x)
        if m.sum() >= 3:
            feats[f"{name}_mean"] = x[m].mean(); feats[f"{name}_std"] = x[m].std()
            feats[f"{name}_min"] = x[m].min(); feats[f"{name}_max"] = x[m].max()
            feats[f"{name}_slope"] = np.polyfit(t[m], x[m], 1)[0]
        else:
            for s in ("mean", "std", "min", "max", "slope"):
                feats[f"{name}_{s}"] = np.nan
    ear = A[:, SIGNALS.index("ear")] if T else np.array([])
    feats["perclos"] = float((ear[~np.isnan(ear)] < ear_closed).mean()) if (~np.isnan(ear)).any() else np.nan
    return feats

def aggregate(probs, k_min=3):
    if len(probs) < k_min:
        return None
    p = np.asarray(probs)
    return {"mean": float(p.mean()), "share_engaged": float((p >= 0.5).mean()), "n": len(p)}


In [ ]:
%%writefile /content/ed_pipeline.py
"""Index, audit and resumable extraction for DAiSEE."""
import os, re, glob, time
from pathlib import Path
import numpy as np, pandas as pd

VIDEO_EXT = {".avi", ".mp4", ".mov", ".mkv"}
_FACTORY = None; _LM = None; _SEQDIR = None

def build_index(root):
    """Return DataFrame: clip, path, split, subject, Boredom/Engagement/Confusion/Frustration."""
    vids = {}
    for dp, _, fs in os.walk(root):
        for f in fs:
            if Path(f).suffix.lower() in VIDEO_EXT:
                vids[Path(f).stem] = os.path.join(dp, f)
    labs = []
    for p in glob.glob(os.path.join(root, "**", "*.csv"), recursive=True):
        n = os.path.basename(p).lower()
        if "label" not in n or "all" in n:
            continue
        split = "Train" if "train" in n else "Validation" if ("valid" in n or "val" in n) else "Test" if "test" in n else None
        if split is None:
            continue
        d = pd.read_csv(p); d.columns = [c.strip() for c in d.columns]; d["split"] = split; labs.append(d)
    if not labs:
        raise RuntimeError("No Train/Validation/Test label CSVs found under " + root)
    L = pd.concat(labs, ignore_index=True)
    L["clip"] = L["ClipID"].astype(str).map(lambda x: Path(x.strip()).stem)
    L["path"] = L["clip"].map(vids)
    L["subject"] = L["clip"].str[:6]
    return L.drop(columns=["ClipID"]), len(vids)

def audit(L, n_videos):
    print(f"video files found: {n_videos}; labelled rows: {len(L)}; rows with a video: {L['path'].notna().sum()}")
    print("\nclips per split:\n", L["split"].value_counts())
    print("\nEngagement level share per split (%):")
    print((pd.crosstab(L["split"], L["Engagement"], normalize="index") * 100).round(2))
    subs = {s: set(L.loc[L.split == s, "subject"]) for s in L.split.unique()}
    names = list(subs)
    print("\nsubjects per split:", {k: len(v) for k, v in subs.items()})
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            print(f"subjects shared by {names[i]} and {names[j]}:", len(subs[names[i]] & subs[names[j]]))
    print("\n(subject = first 6 characters of the clip ID: an assumption, check that the counts above look like ~112 users)")

def _init(factory, seqdir):
    global _LM, _SEQDIR
    _LM = factory(); _SEQDIR = seqdir

def _work(args):
    import ed_features as E
    clip, path = args
    t = time.time()
    try:
        A, det, nc = E.extract_clip(path, _LM)
        seq = np.concatenate([A, det[:, None].astype(float)], axis=1).astype(np.float32)   # signals + raw detection flag
        np.save(os.path.join(_SEQDIR, clip + ".npy"), seq)
        f = E.window_features(A, det); f["clahe_frames"] = nc; f["error"] = ""
    except Exception as e:
        f = {"error": repr(e)[:200]}
    f["clip"] = clip; f["secs"] = time.time() - t
    return f

def run_extraction(L, out_dir, factory, workers=2, chunk=100, limit=None):
    """Resumable: each chunk of clips is saved to out_dir/chunk_XXXX.csv and skipped if already present."""
    import multiprocessing as mp
    os.makedirs(out_dir, exist_ok=True)
    seqdir = os.path.join(out_dir, "seq"); os.makedirs(seqdir, exist_ok=True)
    todo = L.dropna(subset=["path"]).reset_index(drop=True)
    if limit: todo = todo.iloc[:limit]
    n_chunks = (len(todo) + chunk - 1) // chunk
    ctx = mp.get_context("fork")
    with ctx.Pool(workers, initializer=_init, initargs=(factory, seqdir)) as pool:
        for c in range(n_chunks):
            fn = os.path.join(out_dir, f"chunk_{c:04d}.csv")
            if os.path.exists(fn):
                continue
            part = todo.iloc[c * chunk:(c + 1) * chunk]
            t = time.time()
            rows = pool.map(_work, list(zip(part["clip"], part["path"])), chunksize=2)
            pd.DataFrame(rows).to_csv(fn + ".tmp", index=False); os.replace(fn + ".tmp", fn)
            print(f"chunk {c + 1}/{n_chunks} done in {time.time() - t:.0f}s", flush=True)

def collect(L, out_dir):
    F = pd.concat([pd.read_csv(p, dtype={"clip": str}) for p in sorted(glob.glob(os.path.join(out_dir, "chunk_*.csv")))], ignore_index=True)
    F["error"] = F["error"].fillna("")
    return L.drop(columns=["path"]).merge(F, on="clip", how="inner")


In [ ]:
%%writefile /content/ed_models.py
"""Baselines on the clip feature table. Reports accuracy, macro-F1 and per-class recall."""
import numpy as np, pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.utils.class_weight import compute_sample_weight

NON_FEATURE = {"clip", "split", "subject", "Boredom", "Engagement", "Confusion", "Frustration", "error", "secs", "clahe_frames"}

def feature_cols(F):
    return [c for c in F.columns if c not in NON_FEATURE and pd.api.types.is_numeric_dtype(F[c])]

def make_xy(F, task):
    y = F["Engagement"].astype(int)
    if task == "binary":
        y = (y >= 2).astype(int)
    return y

def run_baselines(F, task="four", min_detect=0.0, seed=0):
    from xgboost import XGBClassifier
    F = F[(F["error"] == "") & (F["detect_rate"] >= min_detect)]
    cols = feature_cols(F)
    tr, va, te = (F[F.split == s] for s in ("Train", "Validation", "Test"))
    med = tr[cols].median()
    X = {k: d[cols].fillna(med).fillna(0).values for k, d in (("val", va), ("test", te), ("train", tr))}
    ytr = make_xy(tr, task).values; classes = sorted(set(ytr)); remap = {c: i for i, c in enumerate(classes)}
    labels = [0, 1] if task == "binary" else [0, 1, 2, 3]
    models = {
        "majority": DummyClassifier(strategy="most_frequent"),
        "logistic_regression": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=3000)),
        "xgboost": XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                                 random_state=seed, n_jobs=2, eval_metric="mlogloss"),
    }
    out, fitted = [], {}
    for name, m in models.items():
        yt = np.array([remap[v] for v in ytr])
        if name == "xgboost":
            m.fit(X["train"], yt, sample_weight=compute_sample_weight("balanced", yt))
        else:
            m.fit(X["train"], yt)
        fitted[name] = m
        for split, d in (("Validation", va), ("Test", te)):
            y = make_xy(d, task).values
            p = np.array([classes[i] for i in m.predict(X["val" if split == "Validation" else "test"])])
            row = {"model": name, "split": split, "n": len(y), "accuracy": accuracy_score(y, p),
                   "macro_F1": f1_score(y, p, labels=labels, average="macro", zero_division=0)}
            for l, r in zip(labels, recall_score(y, p, labels=labels, average=None, zero_division=0)):
                row[f"recall_{l}"] = r
            out.append(row)
    return pd.DataFrame(out).round(3), fitted, cols


## 4. Get DAiSEE onto the Colab disk
**Do not download it to your laptop.** Open your approved DAiSEE link in Drive and use **Add shortcut to Drive** (a shortcut uses no storage; *Make a copy* would need 14 GB of your 15 GB quota). Then set `ZIP_ON_DRIVE` to the shortcut's path. If you cannot make a shortcut, paste the file ID from the share link into `FILE_ID` instead.
The zip is 14 GB and the extracted videos need about as much again; Colab's disk is large enough, but it is wiped when the runtime resets.

In [ ]:
import os
ZIP_ON_DRIVE = '/content/drive/MyDrive/DAiSEE.zip'   # path of the shortcut or copy in your Drive
FILE_ID = ''                                           # alternative: file id from the share link

if os.path.exists('/content/daisee') and os.listdir('/content/daisee'):
    print('already extracted')
else:
    if os.path.exists(ZIP_ON_DRIVE):
        os.system(f'cp "{ZIP_ON_DRIVE}" /content/DAiSEE.zip')
    elif FILE_ID:
        os.system('pip -q install gdown'); os.system(f'gdown --id {FILE_ID} -O /content/DAiSEE.zip')
    else:
        raise SystemExit('Set ZIP_ON_DRIVE or FILE_ID first')
    os.system('unzip -q -o /content/DAiSEE.zip -d /content/daisee && rm /content/DAiSEE.zip')
os.system('df -h /content | tail -1')
for dp, dn, fn in os.walk('/content/daisee'):
    if dp.count(os.sep) <= 4: print(dp, '->', len(dn), 'folders,', len(fn), 'files')

## 5. Index and audit (class counts, splits, subject overlap)

In [ ]:
import sys; sys.path.insert(0, '/content')
import pandas as pd, ed_pipeline as P
L, n_videos = P.build_index('/content/daisee')
P.audit(L, n_videos)
L.drop(columns=['path']).to_csv(f'{WORK}/label_index.csv', index=False)

## 6. Smoke test on 20 clips (check detection rate and speed before the full run)

In [ ]:
import shutil, time

def make_landmarker():
    import mediapipe as mp
    from mediapipe.tasks import python as mpp
    from mediapipe.tasks.python import vision
    opts = vision.FaceLandmarkerOptions(
        base_options=mpp.BaseOptions(model_asset_path='/content/face_landmarker.task'),
        running_mode=vision.RunningMode.IMAGE, num_faces=1,
        output_face_blendshapes=True, output_facial_transformation_matrixes=True)
    return vision.FaceLandmarker.create_from_options(opts)

shutil.rmtree(f'{WORK}/smoke', ignore_errors=True)
t = time.time()
P.run_extraction(L, f'{WORK}/smoke', make_landmarker, workers=2, chunk=10, limit=20)
S = P.collect(L, f'{WORK}/smoke')
dt = time.time() - t
print(f'{dt:.0f}s for {len(S)} clips -> about {dt / len(S):.1f} s/clip')
print('errors:', (S.error != '').sum(), '| mean face-detection rate:', S.detect_rate.mean().round(2))
print(f'estimated full run: {dt / len(S) * L.path.notna().sum() / 3600:.1f} hours')
S[['clip', 'split', 'Engagement', 'detect_rate', 'yaw_mean', 'ear_mean', 'perclos', 'motion_mean']].head(10)

**Read the smoke-test result before continuing.** If errors > 0 or the detection rate is near 0, stop and send me the printed output. The head-pose and gaze sign conventions also need a sanity check on real footage: look at `yaw_mean` for a clip where the person turns their head.

## 7. Full extraction (resumable; run again after any disconnect)

In [ ]:
P.run_extraction(L, f'{WORK}/chunks', make_landmarker, workers=2, chunk=100)
F = P.collect(L, f'{WORK}/chunks')
F.to_csv(f'{WORK}/features_all.csv', index=False)
print(F.shape); print(F.groupby('split').detect_rate.mean().round(3)); print('errors:', (F.error != '').sum())

## 8. Baselines: majority, logistic regression, XGBoost (headline metric: macro-F1)

In [ ]:
import ed_models as M
res = {}
for task in ('four', 'binary'):
    res[task], fitted, cols = M.run_baselines(F, task=task)
    print(f'\n=== {task}-level task ===')
    print(res[task].to_string(index=False))
    res[task].to_csv(f'{WORK}/baseline_results_{task}.csv', index=False)

Fill the real numbers into Table 7 of the report only from this output. Next steps: GRU on the per-frame sequences, SHAP on XGBoost, CLAHE on/off perturbation test, and the live aggregate demo.